# Unit 4 — CartPole-v1: REINFORCE (policy gradient) from scratch in PyTorch

**Pass criterion:** `mean_reward − std_reward ≥ 350` (maximum possible score is 500)

No RL library: **we write the policy network and the REINFORCE algorithm ourselves.**

**How to run:** `Runtime → Run all` (~5–15 min, no GPU). The Hugging Face login reads the `HF_TOKEN` Colab secret.

In [ ]:
!pip install -q gymnasium pygame huggingface_hub imageio imageio-ffmpeg

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ENV_ID = "CartPole-v1"
REPO_ID = f"{HF_USERNAME}/Reinforce-{ENV_ID}"
THRESHOLD = 350

import copy, datetime, json, shutil, tempfile
from collections import deque
from pathlib import Path
import imageio
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym

import base64
from IPython.display import HTML, display

def show_video(path, width=500):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 1. The environment
- **Observation (4 numbers):** cart position, cart velocity, pole angle, pole angular velocity.
- **Action (2):** push left / push right.
- **Reward:** **+1** for every step the pole stays upright. The episode ends if the pole tilts more than 12° or the cart leaves the screen. Max 500 steps → max score 500.

In [ ]:
env = gym.make(ENV_ID)
s_size = int(env.observation_space.shape[0])
a_size = int(env.action_space.n)
print("State size:", s_size, "| Action size:", a_size)

## 2. Policy network
`state (4) → Linear → ReLU → Linear → softmax → probabilities of the 2 actions`

`act()` **samples** from these probabilities and also returns `log π(a|s)` of the chosen action (needed for the loss).

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super().__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return F.softmax(self.fc2(x), dim=1)

    def act(self, state):
        state = torch.from_numpy(np.asarray(state, dtype=np.float32)).unsqueeze(0)
        probs = self.forward(state)
        m = Categorical(probs)
        action = m.sample()               # sample, NOT argmax -> exploration
        return action.item(), m.log_prob(action)

EVAL_SEEDS = list(range(1000, 1000 + 100))   # fixed evaluation games -> reproducible score

def evaluate_agent(policy, n_eval_episodes, max_t):
    """Plays n fixed games (same seeds every time) with the stochastic policy."""
    rng_state = torch.get_rng_state()      # don't disturb the training randomness
    torch.manual_seed(0)
    env = gym.make(ENV_ID)
    rewards = []
    for i in range(n_eval_episodes):
        state, _ = env.reset(seed=EVAL_SEEDS[i])
        total = 0.0
        for _ in range(max_t):
            action, _ = policy.act(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            if terminated or truncated:
                break
        rewards.append(total)
    env.close()
    torch.set_rng_state(rng_state)
    return float(np.mean(rewards)), float(np.std(rewards))

def record_video(policy, path, seed=0, fps=30, scale=1, max_t=5000):
    torch.manual_seed(seed)
    env = gym.make(ENV_ID, render_mode="rgb_array")
    state, _ = env.reset(seed=seed)
    frames, total = [env.render()], 0.0
    for _ in range(max_t):
        action, _ = policy.act(state)
        state, reward, terminated, truncated, _ = env.step(action)
        total += reward
        frames.append(env.render())
        if terminated or truncated:
            break
    env.close()
    if scale > 1:
        frames = [np.kron(f, np.ones((scale, scale, 1), dtype=np.uint8)) for f in frames]
    imageio.mimsave(path, frames, fps=fps, macro_block_size=1)
    print(f"Episode reward: {total:.1f}  ({len(frames) - 1} steps)")
    return path

## 3. Watch before training (random weights)

In [ ]:
torch.manual_seed(0)
policy = Policy(s_size, a_size, 16)
show_video(record_video(policy, "untrained.mp4"))

## How REINFORCE works
1. The policy network outputs a **probability** for each action (softmax).
2. The action is **sampled** from these probabilities (`Categorical(...).sample()`), not `argmax` → exploration comes for free.
3. When the episode ends, the **return** of every step is computed backwards: `G_t = r_t + γ·G_(t+1)`.
4. Returns are standardised (zero mean): better-than-average actions get +, worse ones get −.
5. Loss: `L = − Σ log π(a_t|s_t) · G_t` → `loss.backward()` → probabilities of good actions go up.

**Extra safeguard:** every N episodes we evaluate on **10 fixed episodes** (always the same start states) and **keep the best weights**. REINFORCE is very noisy, so the last weights are not always the best. The best weights are pushed to the Hub, and thanks to the fixed seeds the published score matches the one shown here.

In [ ]:
def reinforce(policy, optimizer, hp, eval_every, target, min_ok=None, max_minutes=None):
    """REINFORCE (Monte Carlo policy gradient) with best-model tracking."""
    import time
    t0 = time.time()
    scores_deque = deque(maxlen=100)
    best_score, best_state = -np.inf, None
    history = []
    for i_episode in range(1, hp["n_training_episodes"] + 1):
        saved_log_probs, rewards = [], []
        state, _ = env.reset()
        # 1) Play one full episode with the current policy
        for t in range(hp["max_t"]):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            if terminated or truncated:
                break
        scores_deque.append(sum(rewards))

        # 2) Discounted return G_t for every step, computed backwards: G_t = r_t + gamma * G_{t+1}
        returns = deque(maxlen=hp["max_t"])
        for t in reversed(range(len(rewards))):
            g_next = returns[0] if len(returns) > 0 else 0
            returns.appendleft(hp["gamma"] * g_next + rewards[t])

        # 3) Standardize returns (above-average actions -> pushed up, below-average -> pushed down)
        returns = torch.tensor(returns)
        eps = np.finfo(np.float32).eps.item()
        returns = (returns - returns.mean()) / (returns.std() + eps) if len(returns) > 1 else returns * 0

        # 4) Loss = - sum( log pi(a_t|s_t) * G_t )  -> gradient ascent on expected return
        policy_loss = torch.cat([-lp * g for lp, g in zip(saved_log_probs, returns)]).sum()
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        # 5) Every `eval_every` episodes: evaluate on fixed games, keep the best policy
        if i_episode % eval_every == 0:
            m, s = evaluate_agent(policy, hp["n_evaluation_episodes"], hp["max_t"])
            history.append((i_episode, np.mean(scores_deque), m - s))
            flag = ""
            if m - s > best_score:
                best_score, best_state, flag = m - s, copy.deepcopy(policy.state_dict()), "  <- new best"
            print(f"Episode {i_episode:6d} | train avg(100) {np.mean(scores_deque):7.2f} | "
                  f"eval {m:7.2f} +/- {s:6.2f} -> score {m - s:7.2f}{flag}")
            if best_score >= target:
                print(f"Target reached (score >= {target}). Stopping early.")
                break
            if max_minutes and (time.time() - t0) / 60 > max_minutes and best_score >= min_ok:
                print(f"{max_minutes} min passed and best score {best_score:.2f} >= {min_ok}. Stopping.")
                break
    policy.load_state_dict(best_state)
    return history, best_score

## 4. Training
- `h_size=16`: hidden layer size.
- `gamma=1.0`: no discounting — every step survived is worth the same.
- `lr=1e-2`: learning rate.

In [ ]:
cartpole_hyperparameters = {
    "h_size": 16,
    "n_training_episodes": 2000,
    "n_evaluation_episodes": 10,
    "max_t": 1000,
    "gamma": 1.0,
    "lr": 1e-2,
    "env_id": ENV_ID,
    "state_space": s_size,
    "action_space": a_size,
}
hp = cartpole_hyperparameters
torch.manual_seed(42)
policy = Policy(s_size, a_size, hp["h_size"])
optimizer = optim.Adam(policy.parameters(), lr=hp["lr"])
history, best = reinforce(policy, optimizer, hp, eval_every=50, target=500)
print("Best score (mean - std):", round(best, 2))

In [ ]:
import matplotlib.pyplot as plt
ep, train_avg, sc = zip(*history)
plt.figure(figsize=(8, 3.5))
plt.plot(ep, train_avg, label="train avg (last 100)")
plt.plot(ep, sc, label="eval score (mean - std)")
plt.axhline(THRESHOLD, color="gray", ls="--", label="threshold")
plt.xlabel("episode"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 5. Score (10 fixed episodes)

In [ ]:
mean_reward, std_reward = evaluate_agent(policy, hp["n_evaluation_episodes"], hp["max_t"])
score = mean_reward - std_reward
print(f"mean_reward = {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"score (mean - std) = {score:.2f}  ->  {'PASSED ✅' if score >= THRESHOLD else 'NOT YET ❌ (Run all again: a new seed often fixes it)'}")

## 6. Watch the trained agent

In [ ]:
show_video(record_video(policy, "replay.mp4", seed=0))

## 7. Push to the Hugging Face Hub

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.repocard import metadata_eval_result, metadata_save

def push_to_hub(repo_id, policy, hyperparameters, video_path):
    api = HfApi()
    api.create_repo(repo_id=repo_id, exist_ok=True)
    repo_name = repo_id.split("/")[1]
    env_id = hyperparameters["env_id"]
    mean_reward, std_reward = evaluate_agent(policy, hyperparameters["n_evaluation_episodes"], hyperparameters["max_t"])
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        torch.save(policy, tmp / "model.pt")
        json.dump(hyperparameters, open(tmp / "hyperparameters.json", "w"), default=lambda o: o.item())
        json.dump({"env_id": env_id, "mean_reward": mean_reward, "std_reward": std_reward,
                   "n_evaluation_episodes": hyperparameters["n_evaluation_episodes"],
                   "eval_datetime": datetime.datetime.now().isoformat()},
                  open(tmp / "results.json", "w"))
        metadata = {"tags": [env_id, "reinforce", "reinforcement-learning", "custom-implementation", "deep-rl-class"]}
        metadata.update(metadata_eval_result(
            model_pretty_name=repo_name, task_pretty_name="reinforcement-learning",
            task_id="reinforcement-learning", metrics_pretty_name="mean_reward",
            metrics_id="mean_reward", metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
            dataset_pretty_name=env_id, dataset_id=env_id))
        readme = tmp / "README.md"
        readme.write_text(f"# **Reinforce** Agent playing **{env_id}**\n"
                          f"This is a trained model of a **Reinforce** agent playing **{env_id}**.\n"
                          "Built for Unit 4 of the Hugging Face Deep RL Course.\n", encoding="utf-8")
        metadata_save(readme, metadata)
        shutil.copy(video_path, tmp / "replay.mp4")
        api.upload_folder(repo_id=repo_id, folder_path=tmp, path_in_repo=".")
    print(f"Done: https://huggingface.co/{repo_id}  ({mean_reward:.2f} +/- {std_reward:.2f})")

push_to_hub(REPO_ID, policy, hp, "replay.mp4")

## 8. Check
Check-my-progress → **Unit 4 (CartPole)** ✅

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")

check_like_certificate("CartPole-v1", "reinforce", 350)